# Week 4 Final — PhoBERT v2 + LLM Cascade

Notebook này chỉ giữ luồng final để chạy trên Kaggle:
1. Setup môi trường
2. Pull code mới nhất từ GitHub
3. Load baseline PhoBERT v2 tốt nhất
4. Smoke test cascade trên 10 review
5. Chạy full cascade trên test set


In [ ]:
# Cell 1 — Kaggle GPU + dependencies
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu} | VRAM: {vram:.1f} GB')
else:
    raise RuntimeError('Khong co GPU. Kaggle: Settings -> Accelerator -> GPU')

torch.cuda.empty_cache()
print(f'PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}')

!pip install -q --upgrade pip
!pip install -q transformers==4.41.2 sentence-transformers==2.7.0 tokenizers==0.19.1 accelerate==0.30.1 underthesea py_vncorenlp tabulate tqdm scikit-learn sentencepiece faiss-cpu
!pip install -q openai google-generativeai
print('Dependencies installed')
print('If Kaggle still keeps old packages in memory, restart session and run all cells again.')


In [ ]:
# Cell 2 — Clone/pull latest repo
import os, sys

REPO_URL = 'https://github.com/vudinhminh08/NLP-project-master-study.git'
REPO_BRANCH = 'master'
PROJECT_DIR = '/kaggle/working/absa-project'

if not os.path.exists(PROJECT_DIR):
    !git clone --branch {REPO_BRANCH} --depth=1 {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull origin {REPO_BRANCH}

os.chdir(PROJECT_DIR)
print(f'Working dir: {os.getcwd()}')

for p in ['code/week1', 'code/week2', 'code/week3', 'code/week3_part2']:
    if p not in sys.path:
        sys.path.insert(0, p)

for f in [
    'data/train_preprocessed.csv',
    'data/dev_preprocessed.csv',
    'data/test_preprocessed.csv',
    'outputs/eda/class_weights.json',
    'outputs/results/week2_results_VNcoreNLP/models_cls_only/best_model.pt',
    'outputs/results/week2_results_VNcoreNLP/results_cls_only/week2_test_metrics.json',
]:
    print(f'  [{"OK" if os.path.exists(f) else "MISSING"}] {f}')


In [ ]:
# Cell 3 — Load OpenAI API key from Kaggle Secrets
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
OPENAI_API_KEY = secrets.get_secret('OPENAI_API_KEY')
API_KEY = OPENAI_API_KEY
print('OPENAI_API_KEY loaded from Kaggle Secrets')


In [ ]:
# Cell 4 — Verify final Week 4 files
import os

required_files = [
    'data/train_preprocessed.csv',
    'data/dev_preprocessed.csv',
    'data/test_preprocessed.csv',
    'outputs/eda/class_weights.json',
    'outputs/results/week2_results_VNcoreNLP/models_cls_only/best_model.pt',
    'outputs/results/week2_results_VNcoreNLP/results_cls_only/week2_test_metrics.json',
]

for f in required_files:
    exists = os.path.exists(f)
    size = os.path.getsize(f) if exists else 0
    status = 'OK' if exists else 'MISSING'
    print(f'[{status}] {f} ({size:,} bytes)')

assert all(os.path.exists(f) for f in required_files), 'Missing files for final v2 cascade run'
print('All final-run files verified.')


In [ ]:
# Cell 5 — Setup PhoBERT v2 cascade
import os, sys, json, torch
import pandas as pd
from transformers import AutoTokenizer

REPO_ROOT = os.getcwd()
for p in ['code/week1', 'code/week2', 'code/week3', 'code/week3_part2']:
    full = os.path.join(REPO_ROOT, p)
    if full not in sys.path:
        sys.path.insert(0, full)

from utils.constants import PHOBERT_V2, TRAIN_CONFIG, WEAK_ASPECTS, ZERO_TRAIN_ASPECTS
from utils.helpers import set_seed, save_json
from step4_eval import evaluate_predictions
from model import ABSAPhoBERT
from predict import load_best_model
from rag_retriever import ABSARetriever
from llm_client import LLMClient
from cascade_predictor import run_cascade_on_dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
set_seed(TRAIN_CONFIG['seed'])

BEST_MODEL_NAME = PHOBERT_V2
BEST_ENCODER = 'cls_only'
BEST_CHECKPOINT = 'outputs/results/week2_results_VNcoreNLP/models_cls_only/best_model.pt'
BEST_RESULTS_DIR = 'outputs/results/week2_results_VNcoreNLP'
V2_TEST_METRICS = 'outputs/results/week2_results_VNcoreNLP/results_cls_only/week2_test_metrics.json'
v2_metrics = json.load(open(V2_TEST_METRICS, encoding='utf-8'))

print(f'REPO_ROOT: {REPO_ROOT}')
print(f'Device: {device}')
print(f'Base model: {BEST_MODEL_NAME}')
print(f'Checkpoint: {BEST_CHECKPOINT}')
print(f'Baseline Combined F1: {v2_metrics["macro_combined_f1"]:.4f}')
print(f'WEAK_ASPECTS ({len(WEAK_ASPECTS)}): {WEAK_ASPECTS}')


In [ ]:
# Cell 6 — Smoke test cascade on 10 reviews
BEST_TOKENIZER = AutoTokenizer.from_pretrained(BEST_MODEL_NAME)
BEST_MODEL = ABSAPhoBERT(
    model_name=BEST_MODEL_NAME,
    dropout=TRAIN_CONFIG['dropout'],
    encoder_option=BEST_ENCODER,
).to(device)
BEST_MODEL = load_best_model(BEST_CHECKPOINT, BEST_MODEL, device)

train_df = pd.read_csv('data/train_preprocessed.csv')
test_df = pd.read_csv('data/test_preprocessed.csv')
retriever = ABSARetriever(cache_path='outputs/results/embeddings_cache.npy', use_faiss=True)
retriever.fit(train_df)

api_key = os.environ.get('OPENAI_API_KEY', API_KEY if 'API_KEY' in globals() else '')
if not api_key:
    raise ValueError('Missing OPENAI_API_KEY / API_KEY for cascade.')
llm_client = LLMClient(provider='openai', api_key=api_key)

SMOKE_DIR = os.path.join(BEST_RESULTS_DIR, 'cascade_k4_smoke')
os.makedirs(SMOKE_DIR, exist_ok=True)

y_true_smoke, y_pred_smoke, smoke_stats = run_cascade_on_dataset(
    test_df=test_df,
    train_df=train_df,
    model=BEST_MODEL,
    tokenizer=BEST_TOKENIZER,
    device=device,
    retriever=retriever,
    llm_client=llm_client,
    threshold=0.60,
    k=4,
    sleep_sec=0.0,
    max_samples=10,
    return_records=True,
)

smoke_metrics = evaluate_predictions(
    y_true_smoke,
    y_pred_smoke,
    title='Cascade Smoke Test (10 reviews) — PhoBERT v2',
    exclude_aspects=ZERO_TRAIN_ASPECTS,
    save_path=os.path.join(SMOKE_DIR, 'cascade_smoke_metrics.json'),
)
save_json(smoke_stats, os.path.join(SMOKE_DIR, 'cascade_smoke_stats.json'))

print(f'Base model for cascade: {BEST_MODEL_NAME}')
print(json.dumps(smoke_stats, ensure_ascii=False, indent=2))
print(f"Smoke Combined F1: {smoke_metrics['macro_combined_f1']:.4f}")


In [ ]:
# Cell 7 — Full cascade on test set
CASCADE_DIR = os.path.join(BEST_RESULTS_DIR, 'cascade_k4')
os.makedirs(CASCADE_DIR, exist_ok=True)

y_true_cascade, y_pred_cascade, cascade_stats = run_cascade_on_dataset(
    test_df=test_df,
    train_df=train_df,
    model=BEST_MODEL,
    tokenizer=BEST_TOKENIZER,
    device=device,
    retriever=retriever,
    llm_client=llm_client,
    threshold=0.60,
    k=4,
    sleep_sec=1.0,
    return_records=True,
)

cascade_metrics = evaluate_predictions(
    y_true_cascade,
    y_pred_cascade,
    title='Cascade Test — PhoBERT v2',
    exclude_aspects=ZERO_TRAIN_ASPECTS,
    save_path=os.path.join(CASCADE_DIR, 'cascade_test_metrics.json'),
)
save_json(cascade_stats, os.path.join(CASCADE_DIR, 'cascade_stats.json'))

comparison = {
    'baseline_v2_cls_only': {
        'model': BEST_MODEL_NAME,
        'combined_f1': v2_metrics['macro_combined_f1'],
        'acd_f1': v2_metrics['macro_acd_f1'],
        'spc_f1': v2_metrics['macro_spc_f1'],
    },
    'cascade_v2_rag_k4': {
        'model': BEST_MODEL_NAME,
        'combined_f1': cascade_metrics['macro_combined_f1'],
        'acd_f1': cascade_metrics['macro_acd_f1'],
        'spc_f1': cascade_metrics['macro_spc_f1'],
        'trigger_rate': cascade_stats['trigger_rate'],
        'total_overrides': cascade_stats['total_overrides'],
    },
}
save_json(comparison, os.path.join(CASCADE_DIR, 'baseline_vs_cascade_comparison.json'))

print(f'WEAK_ASPECTS used: {WEAK_ASPECTS}')
print(json.dumps(cascade_stats, ensure_ascii=False, indent=2))
print(json.dumps(comparison, ensure_ascii=False, indent=2))
print(f"Cascade Combined F1: {cascade_metrics['macro_combined_f1']:.4f}")
